# FormCoach - train on the Mendeley multi-view squat videos (Colab / Kaggle)

Everything happens **inside this notebook**, nothing needs to be downloaded to your laptop:

1. download the squat videos of [Mendeley dataset kgbb3yn47p v3](https://data.mendeley.com/datasets/kgbb3yn47p/3) (CC BY 4.0, ~400 MB)
2. run MediaPipe on every video (slowest step, ~20-40 min) and cache the keypoints
3. cut each video into reps and turn every rep into a joint-angle sequence
4. compare the rule baseline with a **1D-CNN** and a **BiLSTM**, cross-validated **by person**
5. test **once** on people that were locked away before any tuning

**Labels are weak:** the dataset only says the whole clip is *good* or *bad* form (26 people, front / side / diagonal views).
Every rep inherits its clip's label, so some "bad" reps are really fine. Expect lower scores than on CFRep
(`train_dl.ipynb`, per-rep judge labels). It tells you "does this rep look like a bad-form set", not *which* fault.

| Platform | Steps |
|---|---|
| **Google Colab** | Runtime > Change runtime type > **GPU (T4)** > Run all. If pip asks you to *restart the session*, restart and Run all again. |
| **Kaggle** | Session options: **Internet ON** and **Accelerator GPU**. Then Run all. |

A GPU matters here: ~2,000 reps x 25 people is ~10x more training than CFRep. Downloads and keypoints are resumable - if
the session dies, just Run all again (on Colab you can also mount Drive, see the last cell).

In [ ]:
# --- 1. Setup: get the repo code (works on Colab, Kaggle or locally) ---
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/Gulshan-heap/form-analyser.git"     # private repo? https://<TOKEN>@github.com/...
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if not os.path.exists("train_dl.py"):                       # not already inside the repo
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".")
    if not os.path.exists("form-analyser"):
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL], capture_output=True, text=True)
        if r.returncode != 0 and IN_KAGGLE:                 # Internet off: use the repo uploaded as a Kaggle Dataset
            found = glob.glob("/kaggle/input/**/train_dl.py", recursive=True)
            assert found, "Turn Internet ON in Session options, or add the repo as a Kaggle Dataset.\n" + r.stderr
            shutil.copytree(os.path.dirname(found[0]), "form-analyser")
        elif r.returncode != 0:
            raise RuntimeError(r.stderr)
    os.chdir("form-analyser")
sys.path.insert(0, os.getcwd())


def sh(cmd):
    """Run a shell command and stream its output into the cell (MediaPipe's log noise filtered out)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not line.startswith(("W0000", "I0000", "WARNING: All log messages")):
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed: {cmd}")


print("working in", os.getcwd())

In [ ]:
# --- 2. Settings ---
PREPARE_DATA = True      # False = skip download/extract/build and reuse datasets/sequences_mendeley.npz
MAX_SUBJECTS = 0         # 0 = all 26 people. Put 4 for a 10-minute smoke test of the whole notebook.
N_TEST_PEOPLE = 5        # people locked away for the final test (chosen randomly with a fixed seed)
FOLDS = 5                # cross-validation groups of people (on the remaining people)
ARCHS = ["cnn", "lstm"]
EPOCHS = 60
SEEDS = 2                # models averaged per fold
BATCH = 64
EXERCISE = "squat"       # the angle channels are squat-specific; other exercises would need new channels

## 3. Download videos, extract keypoints, build sequences
Skipped when `PREPARE_DATA = False`.

In [ ]:
if PREPARE_DATA:
    sh("pip install -q mediapipe")
    sh("python get_model.py")
    sh(f"python get_dataset.py {EXERCISE} --version 3" + (f" --max-subjects {MAX_SUBJECTS}" if MAX_SUBJECTS else ""))

In [ ]:
if PREPARE_DATA:                                           # slowest step: MediaPipe on every frame, resumable
    sh(f"python extract_keypoints.py data/videos/{EXERCISE} --workers {min(4, os.cpu_count() or 2)}")

In [ ]:
if PREPARE_DATA:
    sh(f"python build_sequences.py mendeley --keypoints data/keypoints/{EXERCISE} --videos data/videos/{EXERCISE}")

In [ ]:
# --- 4. Imports and device ---
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import train_dl as T                                        # all model / metric code lives here

FORCE_CPU = False        # set True if a GPU error appears (e.g. an old Kaggle P100)
if FORCE_CPU:
    T.DEVICE = "cpu"
print("torch", torch.__version__, "| device:", T.DEVICE,
      "|", torch.cuda.get_device_name(0) if T.DEVICE == "cuda" else "CPU only - training will be slow, switch on a GPU")

## 5. The data
`invalid` here means the clip was labelled **bad** form.

In [ ]:
d, x, dur, y = T.load("datasets/sequences_mendeley.npz")
person, view, yn = d["person"], d["view"], d["y"]
df = pd.DataFrame(dict(person=person, view=view, clip=d["clip"], bad=yn))
print(f"{len(yn)} reps from {len(set(person))} people, {df["clip"].nunique()} clips, {int(yn.sum())} bad / {int((yn == 0).sum())} good")
print("reps per clip:", df.groupby("clip").size().describe()[["min", "50%", "max"]].to_dict(), "(expect roughly 10-16)")
print(pd.crosstab(df.view, df.bad.map({0: "good", 1: "bad"}), margins=True))

In [ ]:
t = np.linspace(0, 100, x.shape[-1])
for lab, name, c in ((0, "good", "tab:green"), (1, "bad", "tab:red")):
    k = d["x"][yn == lab, 0] * 180
    plt.plot(t, k.mean(0), c=c, label=f"{name} (n={len(k)})")
    plt.fill_between(t, k.mean(0) - k.std(0), k.mean(0) + k.std(0), color=c, alpha=.15)
plt.xlabel("% of rep"); plt.ylabel("knee angle (deg)"); plt.title("Knee angle through a rep"); plt.legend(); plt.show()

## 6. Lock the test people
These people are **never** used for training or choosing settings. Cross-validation (next section) uses only the others.

In [ ]:
test_mask = T.split_people(person, N_TEST_PEOPLE, seed=0)
train_mask = ~test_mask
print("locked test people:", sorted(set(person[test_mask])))
print("train/CV people   :", len(set(person[train_mask])), "->", int(train_mask.sum()), "reps; test:", int(test_mask.sum()), "reps")
sub = lambda m: ({k: d[k][m] for k in ("x", "y", "person", "view", "dur", "min_knee", "max_hbk")}, x[m], dur[m], y[m])
dtr, xtr, durtr, ytr = sub(train_mask)
dte, xte, durte, yte = sub(test_mask)

## 7. Cross-validation by person (train people only)
Use this table to choose the architecture - **not** the test people.

In [ ]:
cv = {}
for name, key, low in (("Rule: hip below knee", "max_hbk", True), ("Rule: knee angle", "min_knee", False)):
    cv[name] = T.rule_baseline(dtr, key, low, FOLDS)
for arch in ARCHS:
    print("cross-validating", arch.upper(), "...", end=" ")
    prob = T.lopo_probs(arch, xtr, durtr, ytr, dtr["person"], EPOCHS, SEEDS, folds=FOLDS, batch=BATCH)
    cv[f"DL: {arch.upper()}"] = (prob >= 0.5, prob)
    print("done")


def table_for(results, yv, vv):
    rows = []
    for name, (pred, score) in results.items():
        m = T.metrics(yv, pred, score)
        row = {"method": name, **{k: round(m[k], 3) for k in ("precision", "recall", "f1", "balanced_acc", "auc")}}
        for v in ("front", "diagonal", "side"):
            row[f"F1 {v}"] = round(T.f1(yv[vv == v], pred[vv == v]), 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("method")


cv_table = table_for(cv, dtr["y"], dtr["view"])
os.makedirs("results", exist_ok=True)
cv_table.to_csv("results/mendeley_cv.csv")
cv_table

In [ ]:
fig, ax = plt.subplots(1, len(cv), figsize=(3.2 * len(cv), 3.2))
for a, (name, (pred, score)) in zip(np.atleast_1d(ax), cv.items()):
    m = T.metrics(dtr["y"], pred, score)
    a.imshow([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], cmap="Blues")
    for i, j, v in ((0, 0, m["tn"]), (0, 1, m["fp"]), (1, 0, m["fn"]), (1, 1, m["tp"])):
        a.text(j, i, v, ha="center", va="center", fontsize=13)
    a.set_xticks([0, 1], ["pred good", "pred bad"]); a.set_yticks([0, 1], ["good", "bad"]); a.set_title(name, fontsize=9)
plt.tight_layout(); plt.show()
cv_table[["F1 front", "F1 diagonal", "F1 side"]].T.plot.bar(figsize=(7, 3), rot=0, title="CV F1 for bad-form reps by camera view")
plt.ylabel("F1"); plt.show()

## 8. Final test on the locked people (run once)
Best deep model by cross-validation F1 is trained on **all** train people and scored on the test people.
Rule thresholds are also learned on the train people only.

In [ ]:
best = max((n for n in cv if n.startswith("DL")), key=lambda n: T.f1(dtr["y"], cv[n][0]))
arch = best.split(": ")[1].lower()
print("chosen by cross-validation:", best)

models = [T.train_one(arch, xtr, durtr, ytr, EPOCHS, seed, batch=BATCH) for seed in range(SEEDS)]
prob_te = np.mean([T.predict(m, xte, durte) for m in models], 0)

final = {best.replace("DL: ", "DL: ") + " (test)": (prob_te >= 0.5, prob_te)}
for name, key, low in (("Rule: hip below knee", "max_hbk", True), ("Rule: knee angle", "min_knee", False)):
    s = dtr[key]
    thr = T.best_threshold(s, dtr["y"], low)
    pred = (dte[key] <= thr) if low else (dte[key] >= thr)
    final[name + " (test)"] = (pred, -dte[key] if low else dte[key])

test_table = table_for(final, dte["y"], dte["view"])
test_table.to_csv("results/mendeley_test.csv")
test_table

In [ ]:
per_person = pd.DataFrame(dict(person=dte["person"], truth=dte["y"], pred=(prob_te >= 0.5).astype(int)))
print("per test person (DL): reps, bad reps, F1")
for p, g in per_person.groupby("person"):
    print(f"  {p}: {len(g):>3} reps, {int(g.truth.sum()):>3} bad, F1 {T.f1(g.truth.values, g.pred.values.astype(bool)):.2f}")

## 9. Where does it go wrong?
Remember the labels are per clip, so a "wrong" good-looking rep inside a bad clip may be the label's fault, not the model's.

In [ ]:
err = pd.DataFrame(dict(clip=d["clip"][test_mask], rep=d["idx"][test_mask], person=dte["person"], view=dte["view"],
                        truth=np.where(dte["y"], "bad", "good"), P_bad=prob_te.round(2),
                        pose_found=d["found"][test_mask].round(2)))[(prob_te >= 0.5) != (dte["y"] == 1)]
print(len(err), "wrong reps out of", len(dte["y"]))
err.sort_values("P_bad").head(30)

## 10. Save
Trains one final model on **all 26 people** (use it for the demo, not for the reported numbers) and saves results.
Commit `results/`, `models/` and `datasets/sequences_mendeley.npz` (a few MB) so nobody has to rerun the slow steps.

In [ ]:
final_model = T.train_one(arch, x, dur, y, EPOCHS, 0, batch=BATCH)
os.makedirs("models", exist_ok=True)
path = f"models/analyser_b_mendeley_{arch}.pt"
torch.save(dict(arch=arch, state=final_model.state_dict(), channels=d["channels"].tolist(), seq_len=x.shape[-1]), path)
print("saved", path)

if IN_COLAB:
    from google.colab import files
    for f in (path, "results/mendeley_cv.csv", "results/mendeley_test.csv", "datasets/sequences_mendeley.npz"):
        files.download(f)
elif IN_KAGGLE:
    print("Kaggle: click 'Save Version' > Save & Run All; files appear under Output (right panel).")

In [ ]:
# Optional (Colab): keep the slow-to-make files on Google Drive so the next session can skip steps 3.
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r data/keypoints /content/drive/MyDrive/formcoach_keypoints      # save
# !cp -r /content/drive/MyDrive/formcoach_keypoints data/keypoints      # restore (then PREPARE_DATA still re-downloads videos; set it False after build)